In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "Dataset/Missing_Train/processed_train_missing_5.csv"
)

FEATURES = [
    "Setting_1",
    "Setting_2",
    "Sensor_2",
    "Sensor_3",
    "Sensor_4",
    "Sensor_6",
    "Sensor_7",
    "Sensor_8",
    "Sensor_9",
    "Sensor_11",
    "Sensor_12",
    "Sensor_13",
    "Sensor_14",
    "Sensor_15",
    "Sensor_17",
    "Sensor_20",
    "Sensor_21"
]


def create_masked_dataset(df, features):

    data = df.copy()

    mask_columns = []

    for feature in features:

        mask_name = f"{feature}_mask"

        # 1 = observed, 0 = missing
        data[mask_name] = (
            data[feature]
            .notna()
            .astype("float32")
        )

        mask_columns.append(mask_name)

    # Replace missing numerical values with neutral placeholder
    data[features] = (
        data[features]
        .fillna(0.0)
        .astype("float32")
    )

    return data, mask_columns


masked_df, MASK_FEATURES = create_masked_dataset(
    df,
    FEATURES
)

MODEL_FEATURES = FEATURES + MASK_FEATURES

print("Original input features:", len(FEATURES))
print("Mask features:", len(MASK_FEATURES))
print("Total model features:", len(MODEL_FEATURES))

print(
    "NaNs remaining:",
    masked_df[MODEL_FEATURES]
    .isna()
    .sum()
    .sum()
)

Original input features: 17
Mask features: 17
Total model features: 34
NaNs remaining: 0


In [2]:
masked_df[
        [
            "Sensor_2",
            "Sensor_2_mask",
            "Sensor_7",
            "Sensor_7_mask",
            "Sensor_20",
            "Sensor_20_mask"
        ]
    ].head(15)

,Sensor_2,Sensor_2_mask,Sensor_7,Sensor_7_mask,Sensor_20,Sensor_20_mask
0,0.183735,1.0,0.726248,1.0,0.713178,1.0
1,0.283133,1.0,0.628019,1.0,0.666667,1.0
2,0.343374,1.0,0.710145,1.0,0.627907,1.0
3,0.343374,1.0,0.740741,1.0,0.573643,1.0
4,0.349398,1.0,0.668277,1.0,0.589147,1.0
5,0.268072,1.0,0.776167,1.0,0.651163,1.0
6,0.382530,1.0,0.723027,1.0,0.744186,1.0
7,0.406626,1.0,0.644122,1.0,0.643411,1.0
8,0.274096,1.0,0.618357,1.0,0.705426,1.0
9,0.150602,1.0,0.602254,1.0,0.627907,1.0


Implementation

In [3]:
import pandas as pd
import numpy as np

train_missing = pd.read_csv(
    "Dataset/Missing_Train/processed_train_missing_5.csv"
)

test_missing = pd.read_csv(
    "Dataset/Missing_Test/processed_test_missing_5.csv"
)

In [4]:
FEATURES = [
    "Setting_1",
    "Setting_2",
    "Sensor_2",
    "Sensor_3",
    "Sensor_4",
    "Sensor_6",
    "Sensor_7",
    "Sensor_8",
    "Sensor_9",
    "Sensor_11",
    "Sensor_12",
    "Sensor_13",
    "Sensor_14",
    "Sensor_15",
    "Sensor_17",
    "Sensor_20",
    "Sensor_21"
]

TARGET = "RUL"
WINDOW = 30

In [5]:
def create_masked_dataset(df, features):

    data = df.copy()

    mask_features = []

    for feature in features:

        mask_name = feature + "_mask"

        # 1 = observed
        # 0 = missing
        data[mask_name] = (
            data[feature]
            .notna()
            .astype("float32")
        )

        mask_features.append(mask_name)

    # Placeholder only.
    # Mask tells model whether 0 is real or missing.
    data[features] = (
        data[features]
        .fillna(0.0)
        .astype("float32")
    )

    return data, mask_features

In [6]:
train_masked, MASK_FEATURES = create_masked_dataset(
    train_missing,
    FEATURES
)

test_masked, _ = create_masked_dataset(
    test_missing,
    FEATURES
)

MODEL_FEATURES = FEATURES + MASK_FEATURES

print("Input features:", len(MODEL_FEATURES))
# 34

Input features: 34


In [7]:
print(
    train_masked[MODEL_FEATURES]
    .isna()
    .sum()
    .sum()
)

print(
    test_masked[MODEL_FEATURES]
    .isna()
    .sum()
    .sum()
)

0
0


In [10]:
clean_train = pd.read_csv(
    "Dataset/processed_train.csv"
)

np.random.seed(42)

engine_ids = clean_train[
    "Engine_ID"
].unique().copy()

np.random.shuffle(engine_ids)

train_ids = engine_ids[:70]
val_ids = engine_ids[70:85]

In [11]:
train_df_30 = train_masked[
    train_masked["Engine_ID"].isin(train_ids)
].copy()

val_df_30 = train_masked[
    train_masked["Engine_ID"].isin(val_ids)
].copy()

In [12]:
def create_sequences(
    data,
    features,
    target,
    window=30
):

    X = []
    y = []

    for engine_id, engine_data in data.groupby("Engine_ID"):

        engine_data = engine_data.sort_values("Cycle")

        values = (
            engine_data[features]
            .values
            .astype("float32")
        )

        targets = (
            engine_data[target]
            .values
            .astype("float32")
        )

        for i in range(window, len(engine_data)):

            X.append(
                values[i-window:i]
            )

            y.append(
                targets[i]
            )

    return np.array(X), np.array(y)

In [13]:
X_train_30, y_train_30 = create_sequences(
    train_df_30,
    MODEL_FEATURES,
    TARGET,
    WINDOW
)

X_val_30, y_val_30 = create_sequences(
    val_df_30,
    MODEL_FEATURES,
    TARGET,
    WINDOW
)

print(X_train_30.shape)
print(X_val_30.shape)

(12216, 30, 34)
(2720, 30, 34)


BiLSTM

In [14]:
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Bidirectional,
    Dropout,
    Dense
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

tf.keras.backend.clear_session()
tf.random.set_seed(42)

bilstm_30 = Sequential([

    Input(
        shape=(
            WINDOW,
            len(MODEL_FEATURES)
        )
    ),

    Bidirectional(
        LSTM(
            64,
            return_sequences=True
        )
    ),

    Dropout(0.2),

    Bidirectional(
        LSTM(32)
    ),

    Dropout(0.2),

    Dense(
        32,
        activation="relu"
    ),

    Dense(1)
])

bilstm_30.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

C:\Users\Shivam\AppData\Roaming\Python\Python310\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.2) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [15]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

history = bilstm_30.fit(
    X_train_30,
    y_train_30,

    validation_data=(
        X_val_30,
        y_val_30
    ),

    epochs=100,
    batch_size=64,
    shuffle=False,

    callbacks=[
        early_stop,
        reduce_lr
    ],

    verbose=1
)

Epoch 1/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 14s 34ms/step - loss: 4162.1567 - mae: 54.4562 - val_loss: 1836.8373 - val_mae: 38.5333 - learning_rate: 0.0010
Epoch 2/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - loss: 1776.6716 - mae: 37.4373 - val_loss: 1748.6857 - val_mae: 37.1328 - learning_rate: 0.0010
Epoch 3/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - loss: 1775.6550 - mae: 37.2997 - val_loss: 1748.5959 - val_mae: 37.1470 - learning_rate: 0.0010
Epoch 4/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - loss: 1772.5734 - mae: 37.2752 - val_loss: 1748.6232 - val_mae: 37.1290 - learning_rate: 0.0010
Epoch 5/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 34ms/step - loss: 1779.2533 - mae: 37.3555 - val_loss: 1748.4240 - val_mae: 37.1422 - learning_rate: 0.0010
Epoch 6/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - loss: 1765.8800 - mae: 37.2280 - val_loss: 1614.7799 - val_mae: 36.1449 - learning_rate: 0.0010
Epoch 7/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - loss: 1308.8289 - mae: 31.8010 

In [16]:
def create_final_test_sequences(
    test_df,
    features,
    window=30
):

    X = []
    engine_ids = []

    for engine_id, engine_data in test_df.groupby(
        "Engine_ID"
    ):

        engine_data = engine_data.sort_values(
            "Cycle"
        )

        values = (
            engine_data[features]
            .values
            .astype("float32")
        )

        if len(values) >= window:

            X.append(
                values[-window:]
            )

            engine_ids.append(
                engine_id
            )

    return (
        np.array(X),
        np.array(engine_ids)
    )

In [17]:
X_test_30, test_engine_ids = (
    create_final_test_sequences(
        test_masked,
        MODEL_FEATURES,
        WINDOW
    )
)

print(X_test_30.shape)

(100, 30, 34)


In [18]:
bilstm_pred_30 = bilstm_30.predict(
    X_test_30,
    verbose=0
).ravel()

In [19]:
RUL_CAP = 125

rul_df = pd.read_csv(
    "Dataset/RUL_FD001.csv"
)

if "Engine_ID" not in rul_df.columns:

    rul_df["Engine_ID"] = np.arange(
        1,
        len(rul_df) + 1
    )

rul_df["RUL"] = rul_df["RUL"].clip(upper=RUL_CAP)


In [20]:
prediction_df = pd.DataFrame({
    "Engine_ID": test_engine_ids,
    "Predicted_RUL": bilstm_pred_30
})

comparison = prediction_df.merge(
    rul_df,
    on="Engine_ID"
)

comparison.head()

,Engine_ID,Predicted_RUL,RUL
0,1,119.513641,112
1,2,114.510368,98
2,3,84.125992,69
3,4,74.531784,82
4,5,113.072563,91


In [21]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

def nasa_score(actual, predicted):
    """NASA / PHM08 asymmetric RUL scoring function (Saxena et al., 2008).
    Penalizes late predictions (predicted RUL > actual RUL) more heavily
    than early ones, since a late prediction means maintenance would be
    scheduled after the engine could already have failed.
    """
    d = np.asarray(predicted) - np.asarray(actual)
    s = np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)
    return np.sum(s)

actual = comparison["RUL"].values
predicted = comparison["Predicted_RUL"].values

mae = mean_absolute_error(
    actual,
    predicted
)

rmse = np.sqrt(
    mean_squared_error(
        actual,
        predicted
    )
)

r2 = r2_score(
    actual,
    predicted
)

score = nasa_score(
    actual,
    predicted
)

print("5% Missing - BiLSTM")
print("MAE  :", mae)
print("RMSE :", rmse)
print("R\u00b2   :", r2)
print("Score:", score)


5% Missing - BiLSTM
MAE  : 11.808030128479004
RMSE : 15.383218197100677
R²   : 0.8526382446289062
Score: 451.50968066426026


In [22]:
def evaluate(name, actual, predicted):
    predicted = np.asarray(predicted).ravel()
    actual = np.asarray(actual).ravel()

    return {
        "Model": name,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": r2_score(actual, predicted),
        "Score": nasa_score(actual, predicted)
    }

results_30 = []

# Add your existing BiLSTM result
results_30.append(
    evaluate(
        "BiLSTM",
        actual,
        bilstm_pred_30
    )
)


LSTM

In [23]:
from sklearn.preprocessing import StandardScaler

y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(
    y_train_30.reshape(-1, 1)
).ravel()

y_val_scaled = y_scaler.transform(
    y_val_30.reshape(-1, 1)
).ravel()

In [25]:
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)

lstm_30 = Sequential([
    Input(
        shape=(WINDOW, len(MODEL_FEATURES))
    ),

    LSTM(
        64,
        return_sequences=True
    ),

    Dropout(0.2),

    LSTM(32),

    Dropout(0.2),

    Dense(
        32,
        activation="relu"
    ),

    Dense(1)
])

lstm_30.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

In [26]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

history_lstm_30 = lstm_30.fit(
    X_train_30,
    y_train_scaled,

    validation_data=(
        X_val_30,
        y_val_scaled
    ),

    epochs=100,
    batch_size=64,
    shuffle=False,

    callbacks=[
        early_stop,
        reduce_lr
    ],

    verbose=1
)

Epoch 1/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 10s 32ms/step - loss: 0.8286 - mae: 0.7843 - val_loss: 0.7036 - val_mae: 0.7386 - learning_rate: 0.0010
Epoch 2/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - loss: 0.5304 - mae: 0.6052 - val_loss: 0.3442 - val_mae: 0.4876 - learning_rate: 0.0010
Epoch 3/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - loss: 0.3959 - mae: 0.5144 - val_loss: 0.2874 - val_mae: 0.4509 - learning_rate: 0.0010
Epoch 4/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - loss: 0.3514 - mae: 0.4782 - val_loss: 0.2658 - val_mae: 0.4223 - learning_rate: 0.0010
Epoch 5/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - loss: 0.3424 - mae: 0.4686 - val_loss: 0.2652 - val_mae: 0.4241 - learning_rate: 0.0010
Epoch 6/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - loss: 0.3356 - mae: 0.4645 - val_loss: 0.2430 - val_mae: 0.4089 - learning_rate: 0.0010
Epoch 7/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - loss: 0.3216 - mae: 0.4539 - val_loss: 0.2462 - val_mae: 0.4201 - learning_rate

In [27]:
lstm_pred_scaled = lstm_30.predict(
    X_test_30,
    verbose=0
)

lstm_pred_30 = y_scaler.inverse_transform(
    lstm_pred_scaled
).ravel()

In [28]:
print(
    evaluate(
        "LSTM",
        actual,
        lstm_pred_30
    )
)

results_30.append(
    evaluate(
        "LSTM",
        actual,
        lstm_pred_30
    )
)

{'Model': 'LSTM', 'MAE': 11.163142204284668, 'RMSE': np.float64(13.96818141932177), 'R2': 0.8785017132759094, 'Score': np.float64(336.0660447183618)}


GRU

In [29]:
from tensorflow.keras.layers import GRU

tf.keras.backend.clear_session()
tf.random.set_seed(42)

gru_30 = Sequential([
    Input(
        shape=(WINDOW, len(MODEL_FEATURES))
    ),

    GRU(
        64,
        return_sequences=True
    ),

    Dropout(0.2),

    GRU(32),

    Dropout(0.2),

    Dense(
        32,
        activation="relu"
    ),

    Dense(1)
])

gru_30.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss="mse",
    metrics=["mae"]
)

gru_30.fit(
    X_train_30,
    y_train_30,
    validation_data=(X_val_30, y_val_30),
    epochs=100,
    batch_size=64,
    shuffle=False,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        )
    ],
    verbose=1
)

gru_pred_30 = gru_30.predict(
    X_test_30,
    verbose=0
).ravel()

results_30.append(
    evaluate(
        "GRU",
        actual,
        gru_pred_30
    )
)

Epoch 1/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 10s 29ms/step - loss: 5346.4941 - mae: 62.4220 - val_loss: 2979.9050 - val_mae: 47.2696
Epoch 2/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - loss: 2070.1187 - mae: 40.1025 - val_loss: 1761.7932 - val_mae: 37.5740
Epoch 3/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - loss: 1791.8436 - mae: 37.4772 - val_loss: 1749.4128 - val_mae: 37.2498
Epoch 4/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - loss: 1802.4213 - mae: 37.5097 - val_loss: 1749.2190 - val_mae: 37.2380
Epoch 5/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - loss: 1800.8336 - mae: 37.5026 - val_loss: 1749.0280 - val_mae: 37.2245
Epoch 6/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - loss: 1786.3341 - mae: 37.3635 - val_loss: 1749.1178 - val_mae: 37.2316
Epoch 7/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - loss: 1804.8922 - mae: 37.5078 - val_loss: 1749.1382 - val_mae: 37.2339
Epoch 8/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - loss: 1804.4310 - mae: 37.5620 - val_loss: 1748

TCN

In [30]:
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    Dropout,
    Add,
    Activation,
    Lambda,
    Dense
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

def tcn_block(
    x,
    filters,
    kernel_size,
    dilation_rate,
    dropout=0.1
):
    residual = x

    x = Conv1D(
        filters,
        kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        activation="relu"
    )(x)

    x = Dropout(dropout)(x)

    x = Conv1D(
        filters,
        kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        activation="relu"
    )(x)

    x = Dropout(dropout)(x)

    if residual.shape[-1] != filters:
        residual = Conv1D(
            filters,
            1,
            padding="same"
        )(residual)

    x = Add()([x, residual])

    return Activation("relu")(x)

In [31]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)

inputs = Input(
    shape=(WINDOW, len(MODEL_FEATURES))
)

x = tcn_block(inputs, 32, 2, 1)
x = tcn_block(x, 32, 2, 2)
x = tcn_block(x, 32, 2, 4)
x = tcn_block(x, 32, 2, 8)

x = Lambda(
    lambda z: z[:, -1, :]
)(x)

x = Dense(
    32,
    activation="relu"
)(x)

x = Dropout(0.1)(x)

outputs = Dense(1)(x)

tcn_30 = Model(
    inputs,
    outputs
)

tcn_30.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss="mse",
    metrics=["mae"]
)

tcn_30.fit(
    X_train_30,
    y_train_30,
    validation_data=(X_val_30, y_val_30),
    epochs=100,
    batch_size=64,
    shuffle=True,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=15,
            restore_best_weights=True
        )
    ],
    verbose=1
)

tcn_pred_30 = tcn_30.predict(
    X_test_30,
    verbose=0
).ravel()

results_30.append(
    evaluate(
        "TCN",
        actual,
        tcn_pred_30
    )
)

Epoch 1/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 9s 21ms/step - loss: 1915.1641 - mae: 34.4674 - val_loss: 475.3055 - val_mae: 17.6379
Epoch 2/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - loss: 668.8225 - mae: 20.5283 - val_loss: 464.0548 - val_mae: 17.4643
Epoch 3/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 613.4321 - mae: 19.5663 - val_loss: 511.8170 - val_mae: 18.8024
Epoch 4/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 571.4947 - mae: 18.7745 - val_loss: 486.5216 - val_mae: 18.3612
Epoch 5/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 554.2057 - mae: 18.4650 - val_loss: 452.9595 - val_mae: 17.6768
Epoch 6/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 547.3464 - mae: 18.3608 - val_loss: 462.4197 - val_mae: 17.9137
Epoch 7/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 517.3593 - mae: 17.7540 - val_loss: 429.3250 - val_mae: 17.2654
Epoch 8/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - loss: 508.0637 - mae: 17.5032 - val_loss: 482.6704 - val_mae:

Random Forest

In [32]:
X_train_flat_30 = X_train_30.reshape(
    X_train_30.shape[0],
    -1
)

X_test_flat_30 = X_test_30.reshape(
    X_test_30.shape[0],
    -1
)

In [33]:
from sklearn.ensemble import RandomForestRegressor

rf_30 = RandomForestRegressor(
    n_estimators=500,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf_30.fit(
    X_train_flat_30,
    y_train_30
)

rf_pred_30 = rf_30.predict(
    X_test_flat_30
)

results_30.append(
    evaluate(
        "Random Forest",
        actual,
        rf_pred_30
    )
)

XGBoost

In [34]:
from xgboost import XGBRegressor

X_val_flat_30 = X_val_30.reshape(
    X_val_30.shape[0],
    -1
)

xgb_30 = XGBRegressor(
    n_estimators=700,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_30.fit(
    X_train_flat_30,
    y_train_30,
    eval_set=[
        (
            X_val_flat_30,
            y_val_30
        )
    ],
    verbose=False
)

xgb_pred_30 = xgb_30.predict(
    X_test_flat_30
)

results_30.append(
    evaluate(
        "XGBoost",
        actual,
        xgb_pred_30
    )
)

In [35]:
results_30_df = pd.DataFrame(
    results_30
).sort_values("RMSE")

print(results_30_df)

           Model        MAE       RMSE        R2       Score
5        XGBoost  10.490231  13.760837  0.882082  307.510576
1           LSTM  11.163142  13.968181  0.878502  336.066045
3            TCN  11.973289  14.877687  0.862164  297.182293
2            GRU  11.080825  14.975237  0.860351  414.395254
0         BiLSTM  11.808030  15.383218  0.852638  451.509681
4  Random Forest  13.620060  18.041973  0.797298  886.709723
